# 📘 Lesson 16: KNN Regression & The Curse of Dimensionality

Continuous target estimation via distance-weighted average $k$-nearest values. Analyzing the exponential data sparsity and distance metric breakdown in high dimensions.

---
**Interactive Step-by-Step Notebook**: Execute cells sequentially to observe data flow, mathematical transformations, and model evaluations.


### 🔹 1. IMPORT LIBRARIES

**Objective**: Import Core Numerical & Machine Learning Libraries.

- Sets up `numpy` for linear algebra, `pandas` for dataframes, `matplotlib`/`seaborn` for plotting, and `sklearn` estimators.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline


### 🔹 2. LOAD DATA

**Objective**: Dataset Loading & Synthetic Data Generation.

- Ingests or synthesizes sample data. Inspects feature shapes, target distributions, and missing values.


In [ ]:
df = pd.read_csv("/Users/mac/Desktop/Machine Learning/ai-roadmap/content/ml/supervised/11 k nearest neighbour/data/TelcoCustomerChurn.csv")

print("First 5 rows:\n", df.head())
print("\nInfo:\n")
df.info()


### 🔹 3. DATA EXPLORATION

**Objective**: Visual Diagnostics & Decision Boundary Plots.

- Visualizes feature relationships, regression curves, decision surfaces, or clustering partitions.


In [ ]:
print("\nTarget distribution:\n", df["Churn"].value_counts())

# Correlation (numerical only)
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


### 🔹 Group analysis

**Objective**: Execution & Utility Transformation.

- Transforms data features, computes intermediate statistics, or runs diagnostic evaluations.


In [ ]:
print("\nChurn by Contract:\n")
print(df.groupby("Contract")["Churn"].value_counts(normalize=True))



# 4. DATA CLEANING


### 🔹 Fix TotalCharges

**Objective**: Execution & Utility Transformation.

- Transforms data features, computes intermediate statistics, or runs diagnostic evaluations.


In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Drop unnecessary column safely
df.drop("customerID", axis=1, inplace=True, errors="ignore")


### 🔹 Encode categorical variables

**Objective**: Execution & Utility Transformation.

- Transforms data features, computes intermediate statistics, or runs diagnostic evaluations.


In [ ]:
df = pd.get_dummies(df, drop_first=True)


# 5. SPLIT FEATURES & TARGET
X = df.drop("Churn_Yes", axis=1)
y = df["Churn_Yes"]


### 🔹 6. TRAIN-TEST SPLIT

**Objective**: Data Splitting & Feature Normalization.

- Splits data into independent train and test sets to evaluate generalization.

- Scales numerical features to zero mean ($\mu=0$) and unit variance ($\sigma=1$) to prevent features with large scales from dominating gradients or distance metrics.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


### 🔹 7. SCALING

**Objective**: Data Splitting & Feature Normalization.

- Splits data into independent train and test sets to evaluate generalization.

- Scales numerical features to zero mean ($\mu=0$) and unit variance ($\sigma=1$) to prevent features with large scales from dominating gradients or distance metrics.


In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# 8. TRAIN MODEL
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)


### 🔹 9. EVALUATION

**Objective**: Model Inference & Prediction Generation.

- Computes predictions or predicted probability scores on unseen test samples.


In [ ]:
y_pred = knn.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


### 🔹 10. OPTIMIZE K

**Objective**: Model Training & Parameter Optimization.

- Executes optimization (OLS normal equation, gradient descent, tree split search, or centroid convergence) on the training set.


In [ ]:
k_values = range(1, 21)
accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    accuracies.append(accuracy_score(y_test, y_pred))


### 🔹 Plot K vs Accuracy

**Objective**: Visual Diagnostics & Decision Boundary Plots.

- Visualizes feature relationships, regression curves, decision surfaces, or clustering partitions.


In [ ]:
plt.plot(k_values, accuracies, marker='o')
plt.xlabel("K value")
plt.ylabel("Accuracy")
plt.title("K vs Accuracy")
plt.show()


### 🔹 11. CROSS-VALIDATION

**Objective**: Import Core Numerical & Machine Learning Libraries.

- Sets up `numpy` for linear algebra, `pandas` for dataframes, `matplotlib`/`seaborn` for plotting, and `sklearn` estimators.


In [ ]:
scores = []

for k in k_values:
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])
    
    cv_score = cross_val_score(pipeline, X, y, cv=5)
    scores.append(cv_score.mean())

optimal_k = k_values[np.argmax(scores)]
print("\nBest K from Cross-Validation:", optimal_k)


### 🔹 12. DISTANCE METRIC COMPARISON

**Objective**: Model Training & Parameter Optimization.

- Executes optimization (OLS normal equation, gradient descent, tree split search, or centroid convergence) on the training set.


In [ ]:
metrics = ["euclidean", "manhattan", "minkowski"]

for metric in metrics:
    knn = KNeighborsClassifier(n_neighbors=optimal_k, metric=metric)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    print(f"{metric}: {accuracy_score(y_test, y_pred)}")


### 🔹 KNN pipeline

**Objective**: Data Splitting & Feature Normalization.

- Splits data into independent train and test sets to evaluate generalization.

- Scales numerical features to zero mean ($\mu=0$) and unit variance ($\sigma=1$) to prevent features with large scales from dominating gradients or distance metrics.


In [ ]:
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=optimal_k))
])

# Logistic Regression pipeline
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000))
])

knn_scores = cross_val_score(knn_pipeline, X, y, cv=5)
lr_scores = cross_val_score(lr_pipeline, X, y, cv=5)

print("KNN CV Accuracy:", knn_scores.mean())
print("Logistic Regression CV Accuracy:", lr_scores.mean())


## 🎯 Summary & Key Takeaways
1. **Core Insight**: Review the printed parameters, loss curves, and evaluation metrics above.
2. **Best Practice**: Always ensure proper feature scaling, validation splitting, and metric selection tailored to the problem distribution.
3. **Next Steps**: Compare these results with related models in subsequent lessons.
